We're on https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html

In [21]:
import torch

In [22]:
torch.rand(2, 2, 2)

tensor([[[0.7779, 0.3123],
         [0.9820, 0.9876]],

        [[0.4549, 0.1073],
         [0.8404, 0.1485]]])

In [23]:
torch.cuda.is_available()

True

In [24]:
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [25]:
training_data = datasets.FashionMNIST(
    root = "data", # creates a directory to store the data
    train = True,
    download = True,
    transform = ToTensor(),
)

test_data = datasets.FashionMNIST(
    root = "data",
    train = False,
    download = True,
    transform = ToTensor(),
)


In [26]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size = batch_size)
test_dataloader = DataLoader(test_data, batch_size = batch_size)

I think... that the DataLoader splices the input array ("tensor") into as many slices as it can subject to the batch size limitation. So, it returns an interable: a training/testing sub-dataset indexed by "batch number."

In [36]:
len(train_dataloader)

938

In [38]:
len(training_data)

60000

In [39]:
len(train_dataloader) * batch_size

60032

In [27]:
for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


Figure out if we can use the accelerator.

In [28]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Now, define a Torch network.

In [29]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        """
        Define the architecture.
        """
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        """
        How data gets passed through the network during a forward pass.
        """
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [30]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Loss and optimizer:

In [31]:
objective_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

What is in the parameters?

In [32]:
for i in model.parameters():
    print(i)

Parameter containing:
tensor([[ 0.0125,  0.0047,  0.0295,  ..., -0.0354,  0.0210, -0.0138],
        [-0.0207, -0.0349,  0.0279,  ..., -0.0299,  0.0031, -0.0128],
        [ 0.0177, -0.0015,  0.0098,  ..., -0.0155,  0.0256, -0.0210],
        ...,
        [ 0.0226, -0.0346,  0.0191,  ...,  0.0254,  0.0158, -0.0152],
        [-0.0196, -0.0008, -0.0296,  ..., -0.0129,  0.0278,  0.0025],
        [-0.0246, -0.0329,  0.0239,  ..., -0.0195, -0.0016, -0.0259]],
       device='cuda:0', requires_grad=True)
Parameter containing:
tensor([-3.0726e-03,  2.1591e-02,  1.4210e-02,  2.7956e-02,  1.9749e-02,
        -1.2674e-02, -2.4587e-02,  3.1432e-02, -1.4960e-02, -3.1260e-03,
         1.4821e-02,  2.7242e-02,  3.0421e-02, -1.1135e-02, -4.5024e-04,
         2.1335e-02,  2.8272e-02,  2.7861e-02,  2.6067e-02,  1.2274e-02,
         7.6149e-04,  1.9824e-02, -3.2876e-02,  1.5283e-02, -2.9379e-02,
         3.4332e-02,  1.6392e-02,  3.3506e-02,  2.3487e-02, -1.2963e-02,
         3.5332e-02, -1.9059e-02,  2.832

Seems like it's all the weights and biases (weight kernels).

In [40]:
def train(dataloader, model, loss_function, optimizer):
    """
    Define the flow of a *single* training loop.
    """
    size = len(dataloader.dataset)

    model.train()
    
    for batch, (X, y) in enumerate(dataloader):
        
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_function(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [41]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


So, we need two functions that define the training and testing phases.

Now, we try to run some training.

In [42]:
epochs = 5
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train(train_dataloader, model, objective_function, optimizer)
    test(test_dataloader, model, objective_function)
print("Done!")

Epoch 1
-------------------------------
loss: 1.182587  [   64/60000]
loss: 1.180797  [ 6464/60000]
loss: 1.012156  [12864/60000]
loss: 1.133256  [19264/60000]
loss: 1.005460  [25664/60000]
loss: 1.040658  [32064/60000]
loss: 1.067970  [38464/60000]
loss: 1.009838  [44864/60000]
loss: 1.051739  [51264/60000]
loss: 0.982970  [57664/60000]
Test Error: 
 Accuracy: 65.4%, Avg loss: 0.996534 

Epoch 2
-------------------------------
loss: 1.057622  [   64/60000]
loss: 1.076473  [ 6464/60000]
loss: 0.890922  [12864/60000]
loss: 1.037088  [19264/60000]
loss: 0.914221  [25664/60000]
loss: 0.941575  [32064/60000]
loss: 0.986861  [38464/60000]
loss: 0.929007  [44864/60000]
loss: 0.969130  [51264/60000]
loss: 0.913612  [57664/60000]
Test Error: 
 Accuracy: 66.7%, Avg loss: 0.920743 

Epoch 3
-------------------------------
loss: 0.966938  [   64/60000]
loss: 1.005299  [ 6464/60000]
loss: 0.805201  [12864/60000]
loss: 0.969929  [19264/60000]
loss: 0.854601  [25664/60000]
loss: 0.869560  [32064/600

In [44]:
torch.save(model.state_dict(), "model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


In [45]:
model.state_dict()

OrderedDict([('linear_relu_stack.0.weight',
              tensor([[ 0.0125,  0.0047,  0.0295,  ..., -0.0352,  0.0210, -0.0138],
                      [-0.0207, -0.0349,  0.0279,  ..., -0.0302,  0.0030, -0.0128],
                      [ 0.0177, -0.0015,  0.0098,  ..., -0.0155,  0.0256, -0.0210],
                      ...,
                      [ 0.0226, -0.0346,  0.0191,  ...,  0.0257,  0.0158, -0.0152],
                      [-0.0196, -0.0008, -0.0296,  ..., -0.0128,  0.0278,  0.0025],
                      [-0.0246, -0.0329,  0.0239,  ..., -0.0195, -0.0016, -0.0259]],
                     device='cuda:0')),
             ('linear_relu_stack.0.bias',
              tensor([ 2.8138e-03,  2.1794e-02,  1.3565e-02,  3.9767e-02,  2.1446e-02,
                       1.1472e-03, -2.0220e-02,  3.8789e-02, -1.5070e-02, -1.4643e-03,
                       1.3918e-02,  2.8253e-02,  4.0074e-02, -1.2606e-02,  4.1309e-03,
                       3.2604e-02,  2.5267e-02,  2.6677e-02,  3.5757e-02,  1.6632

Interesting.

In [46]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model.pth", weights_only = True))

<All keys matched successfully>

In [48]:
model.eval()

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)